# Stage 8 — NeMo fine-tuning: data and LoRA preparation

This notebook checks the saved Stage 7 data, prepares processor inputs, and
loads NeMo LoRA adapters, then runs one complete training smoke-test step.
Research and educational use only; this is not a clinical diagnostic system.

Run with a Python kernel inside the existing GPU-enabled NeMo AutoModel
26.06.00 container, with the project mounted at `/workspace/skin-lesion-ai`
and the persistent Hugging Face cache at `/root/.cache/huggingface`.
The host Conda environment stays separate. Loading uses cached assets only.

The saved CSVs contain one fixed instruction and abbreviation targets, unlike
the handoff's four-instruction/full-name description. We preserve these CSVs.
Docker changes where files appear; the adapter resolves their old host paths.


In [ ]:
from pathlib import Path
import sys

# Works when launched from the repository root or notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "data/processed/multimodal/train.csv").is_file()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.stage8.dataset import HAM10000MultimodalDataset

print("Project root:", PROJECT_ROOT)


## Read existing splits

The adapter behaves like a PyTorch map-style dataset: `len(dataset)` counts
records and `dataset[index]` loads one RGB image. Images are loaded lazily,
so creating the dataset does not hold thousands of images in memory.

No split is created, no instruction is regenerated, and no image is duplicated.
The test split is unavailable through this development adapter. Existing
lesion-level separation does not establish patient-level independence.


In [ ]:
train = HAM10000MultimodalDataset(PROJECT_ROOT, "train")
val = HAM10000MultimodalDataset(PROJECT_ROOT, "val")
assert len(train) == 7002
assert len(val) == 1532
print(f"Train: {len(train)}; validation: {len(val)}")

# Fixed existing row order gives a deterministic four-image inspection sample.
# This sample is for plumbing checks, not performance estimation.
for index in range(4):
    sample = train[index]
    assert sample.image.mode == "RGB"
    assert sample.split == "train"
    print(sample.image_id, sample.image.size, sample.image_path)


## Separate model input from the answer

A VLM combines image inputs and text. Its processor will later turn these
messages into image tensors and token IDs using the model's chat template.
The dataset adapter does not perform that model-specific processing yet.

During training, an assistant message provides the known classification target.
During generation, the model receives only the user image and instruction;
the answer remains available separately for scoring. Category names appear
in the fixed instruction as possible choices, which is different from telling
the model which choice is correct. We add no medical descriptions or findings.


In [ ]:
sample = train[0]
evaluation_messages = sample.evaluation_messages()
training_messages = sample.training_messages()

assert [message["role"] for message in evaluation_messages] == ["user"]
assert evaluation_messages[0]["content"][1]["text"] == sample.instruction
assert [message["role"] for message in training_messages] == ["user", "assistant"]
assert training_messages[-1]["content"][0]["text"] == sample.response
# Building training messages must not mutate the evaluation prompt.
assert len(evaluation_messages) == 1

print("Instruction:", sample.instruction)
print("Stored target (not part of evaluation input):", sample.response)
print("Evaluation roles:", [message["role"] for message in evaluation_messages])


## Dataset checkpoint

Both split sizes must match, four training images must open under the container
root, and the message checks must pass before continuing.
`ISIC_0026993` is a test image and must not become a development smoke-test case.


## Processor and assistant-only loss

The processor combines a tokenizer with image preprocessing. The chat template
adds role markers; the processor expands image placeholders into model inputs.
For this memory feasibility experiment, disable image splitting and set a
512-pixel longest edge. This reduces image views; it is not a quality-tuned
setting and may discard details. Record it for any future comparison.

A training label of `-100` means “ignore this position in the loss.” Mask the
entire user/image prompt and assistant role prefix. Supervise only the answer
and end-of-turn formatting. The model shifts labels internally to predict the
next token. Never manually shift again. Reject prefix mismatches or oversized
sequences instead of guessing a mask or silently truncating image tokens.


In [ ]:
from src.stage8.processing import (
    load_processor, prepare_evaluation_inputs, SmolVLMTrainingCollator,
)

processor = load_processor()
collator = SmolVLMTrainingCollator(processor)
sample = train[0]
batch = collator([sample])
evaluation_inputs = prepare_evaluation_inputs(sample, processor)
assert "labels" not in evaluation_inputs
prefix_length = evaluation_inputs["input_ids"].shape[1]
assert (batch["labels"][:, :prefix_length] == -100).all()
supervised = batch["labels"][0] != -100
print("Tensor shapes:", {key: tuple(value.shape) for key, value in batch.items()})
print("Supervised completion:", repr(processor.tokenizer.decode(batch["labels"][0, supervised])))


## Load NeMo LoRA adapters

NeMo's generic image-text loader uses the Hugging Face Idefics3 architecture
for SmolVLM. We use its installed native LoRA implementation, with standard
PyTorch SDPA attention and no optional Liger/Triton optimizations at this stage.

LoRA adds a low-rank update to a frozen matrix: `W + (alpha / rank) * B @ A`.
Here rank is 4 and alpha is 8. Attach it only to query/value projections in the
language model's attention layers. The vision encoder, multimodal connector,
and all original weights remain frozen. This limits trainable parameters;
it does not remove activation memory needed for backpropagation.

The installed loader returned FP32 despite the dtype request during inspection.
Our loader explicitly converts the base to FP16 before attaching FP32 adapters.
The later optimizer will update only those adapters under mixed precision.
No quantization is used; this is LoRA, not QLoRA.


In [ ]:
import torch
from src.stage8.model import load_lora_model

# Release a previous notebook model before rerunning this cell.
if "model" in globals():
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
torch.manual_seed(42)
model, lora_report = load_lora_model()
print(lora_report)
print("Current GPU allocation (GiB):", torch.cuda.memory_allocated() / 1024**3)


## Verify a forward loss without updating weights

This single training image checks that the processor tensors and labels are
accepted by the model. `no_grad()` avoids building a backward graph, while
autocast runs suitable operations in FP16 with FP32 adapter storage.
A finite loss proves the forward interface works; it does **not** prove that
backward and optimizer state fit in 4 GB, or that the model has improved.


In [ ]:
model.eval()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    gpu_batch = {key: value.to("cuda") for key, value in batch.items()}
    output = model(**gpu_batch)
    forward_loss = output.loss.item()
    assert torch.isfinite(output.loss).item()
torch.cuda.synchronize()
print("Forward-only loss:", forward_loss)
print("Forward-only peak allocated GiB:", torch.cuda.max_memory_allocated() / 1024**3)
print("Forward-only peak reserved GiB:", torch.cuda.max_memory_reserved() / 1024**3)
del output, gpu_batch


## Run the complete one-step smoke test

The repository runner uses `configs/stage8/smolvlm_lora.yaml`: batch size 1,
a fixed four-image training subset, and one optimizer step on its first image.
It checks finite loss and gradients, frozen base parameters, a real adapter
update, and optimizer state allocation. The initial FP16 loss scale is 128;
the default 65536 caused overflow and a skipped update during the first probe.

Each successful run saves native NeMo adapter tensors and `metrics.json` under
an ignored, timestamped `outputs/stage8/` directory. Adapter tensor serialization
is checked, but this is not a resumable training checkpoint or a Hugging Face
PEFT export. Peak allocated and reserved memory cover the full training step;
they exclude some CUDA driver/library and other-process memory.

The shell equivalent inside the container, from the repository root, is:

```bash
python -m src.stage8.smoke_test --config configs/stage8/smolvlm_lora.yaml
```

The following cell runs a new training experiment. Release the earlier model
first so the notebook does not keep two copies on the 4 GB GPU.


In [ ]:
import gc
from src.stage8.smoke_test import run

if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
smoke_report = run(PROJECT_ROOT / "configs/stage8/smolvlm_lora.yaml", PROJECT_ROOT)
assert smoke_report["status"] == "passed"


## Interpretation

A passing run demonstrates one complete LoRA update on this hardware with the
recorded settings. It does not measure classification quality, guarantee longer
runs fit, or establish clinical validity. No validation/test quality metrics
are computed. Keep test data isolated when choosing subsequent experiments.
